# 02 · Posterior summaries over the sample axis

**What this teaches:** how to read a single number, an interval, an exceedance
probability or a worst-case off the **sample axis** of a `PredictionFrame` — using
`views_frames_summarize` — and, just as important, **when each summary lies**. Every
summary is swept over a *zoo* of distribution shapes typical of the VIEWS estimand, and
checked for *honesty* (calibration / coverage) against a **known latent truth**.

**Audience:** anyone about to report a posterior to a downstream consumer (a dashboard,
an FAO food-security alert) and wanting to trust the number they print.

> **How to read this notebook.** Public, **frozen** `views_frames` /
> `views_frames_summarize` API only, on **synthetic** data with a *known* data-generating
> process (`notebooks/_synthetic.py`) — so we can check whether the intervals actually
> cover. Fixed seeds; a full *Run All* is well under a minute. All plots are **static**
> small-multiples (a deliberate choice — reproducible and cheap; an interactive widget
> tour is out of scope for an un-gated artifact).
>
> Scoring rules live in *views-evaluation*, not here — this notebook **shows** calibration
> to build trust in the summaries; it does not add a scoring API.

### Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

for _p in (Path.cwd(), Path.cwd() / "notebooks"):
    if (_p / "_synthetic.py").exists():
        sys.path.insert(0, str(_p))
        break
import _synthetic as syn

from views_frames import SpatialLevel
from views_frames_summarize import (
    aggregate_distributions,
    bimodality,
    collapse,
    exceedance,
    expected_shortfall,
    hdi,
    hdi_tower,
    map_estimate,
    quantiles,
    tower_point,
)
from views_frames_summarize.conformance import assert_summarizer_contract

np.set_printoptions(precision=3, suppress=True)

## 1 · The estimand, and the philosophy

A VIEWS forecast for a `(month, cell)` is not a number — it is a **posterior of `S`
predictive samples** (MC-dropout × a distributional head × an ensemble, all pooled onto
one sample axis). That posterior is, by the nature of conflict, **low-sample,
zero-inflated, right-skewed, heavy-tailed and occasionally multimodal**.

A *summary* projects that distribution onto something a consumer can act on — a point, an
interval, a probability. **The summary you choose encodes what you care about**, and on
these shapes the choices genuinely disagree: the mean is not the mode, the
highest-density interval is not the equal-tailed one, the worst-case is not the max. The
rest of this notebook makes those disagreements concrete — and asks which summaries are
*honest* on which shapes.

## 2 · The distribution zoo

Six shapes spanning the estimand, each generated with a **known** mode and mean (the red
/ green marks). We show the pooled regime (`S=1024`) and the low-sample regime (`S=32`)
side by side — most of the summaries below behave very differently between the two.

In [ ]:
SHOW = ["zero_inflated", "right_skewed_gamma", "near_symmetric", "heavy_tailed", "bimodal"]

def plot_zoo(n_samples, seed=0):
    zoo = syn.distribution_zoo(n_samples=n_samples, seed=seed)
    fig, axes = plt.subplots(1, len(SHOW), figsize=(13, 2.4))
    for ax, name in zip(axes, SHOW):
        samples, truth = zoo[name]
        ax.hist(samples, bins=30, color="steelblue", density=True)
        ax.axvline(truth.mean, color="crimson", lw=1.5, label="true mean")
        ax.axvline(truth.mode, color="seagreen", lw=1.5, ls="--", label="true mode")
        ax.set_title(name, fontsize=8.5)
        ax.set_yticks([])
    axes[0].legend(fontsize=7)
    fig.suptitle(f"the zoo at S={n_samples}", fontsize=10)
    plt.tight_layout(); plt.show()

plot_zoo(1024)
plot_zoo(32)

## 3 · Point estimates — and why the *mode* is contested

Four ways to collapse a posterior to one number, all on the **frozen** surface:

* `collapse(frame, np.mean)` / `collapse(frame, np.median)` — generic folds (the package
  owns the *mechanism*, you inject the statistic);
* `map_estimate(frame)` — the histogram MAP (mode) as faoapi/reporting compute it;
* `tower_point(frame)` — the tower-tip "shorth" (median of the 50%-mass HDI floor).

On a symmetric posterior these agree. On the skewed, zero-inflated shapes they **don't**,
and the mode is the most contested of all — `map_estimate` carries a documented
low-sample, lowest-bin tie-break **bias toward zero** on right-skewed posteriors
(register C-32); `tower_point` is the unbinned, duplicate-robust alternative.

In [ ]:
zoo = syn.zoo_frames(n_samples=1024, seed=0)
frame = zoo.prediction

est = {
    "true_mode": np.array([t.mode for t in zoo.truths]),
    "true_mean": np.array([t.mean for t in zoo.truths]),
    "map_estimate": map_estimate(frame).values[:, 0],
    "tower_point": tower_point(frame).values[:, 0],
    "mean": collapse(frame, np.mean).values[:, 0],
    "median": collapse(frame, np.median).values[:, 0],
}
print(f"{'shape':24s}" + "".join(f"{k:>13s}" for k in est))
for i, name in enumerate(zoo.names):
    print(f"{name:24s}" + "".join(f"{est[k][i]:13.2f}" for k in est))

**The C-32 bias, made visible.** The histogram MAP's lowest-bin tie-break biases the
mode **toward zero at low `S`** — but it only *bites* when the true mode sits **away from**
zero, where there is room to fall. On `right_skewed_gamma` (true mode 1.8) the `S=32` MAP
drops well below it while the tower tip does not. On shapes whose mode is *already* near zero
(`right_skewed_lognormal`, `heavy_tailed`) a low MAP is indistinguishable from a correct one —
so the bias is a property of the **(shape, `S`)** pair, not of "right-skewed" alone.

In [ ]:
low = syn.zoo_frames(n_samples=32, seed=0)
m_low = map_estimate(low.prediction).values[:, 0]
t_low = tower_point(low.prediction).values[:, 0]
print(f"{'shape':24s}{'true_mode':>12s}{'MAP@S=32':>12s}{'tower@S=32':>12s}")
for i, name in enumerate(low.names):
    if "skewed" in name or name == "heavy_tailed":
        print(f"{name:24s}{low.truths[i].mode:12.2f}{m_low[i]:12.2f}{t_low[i]:12.2f}")

## 4 · The HDI tower

`hdi_tower(frame, masses=(...))` returns nested **highest-density** intervals — the
*shortest* interval holding each mass — built so wider always contains narrower (nested by
construction). On a right-skewed posterior the HDI hugs the dense left and is **not** the
equal-tailed interval (we put those head-to-head in the honesty panel below).

In [ ]:
masses = (0.5, 0.9, 0.95, 0.99)
tower = hdi_tower(frame, masses=masses)         # (N, 4, 2)

fig, axes = plt.subplots(1, len(SHOW), figsize=(13, 2.6), sharey=False)
for ax, name in zip(axes, SHOW):
    i = zoo.names.index(name)
    for j, m in enumerate(masses):
        lo, hi = tower[i, j]
        ax.plot([lo, hi], [m, m], lw=3, solid_capstyle="butt")
    ax.axvline(zoo.truths[i].mode, color="seagreen", ls="--", lw=1)
    ax.set_title(name, fontsize=8.5); ax.set_xlabel("value", fontsize=8)
axes[0].set_ylabel("HDI mass"); axes[0].set_yticks(masses)
fig.suptitle("nested HDI tower (50 / 90 / 95 / 99%); green = true mode", fontsize=10)
plt.tight_layout(); plt.show()

## 5 · Quantiles & exceedance — `P(Y > c)` and the onset flag

`quantiles(frame, qs)` and `exceedance(frame, thresholds)` are index-aligned arrays. The
flagship exceedance is **`P(Y > 0)`** — the probability of *any* activity (onset). It is a
distribution-agnostic counting reducer (no histogram, no unimodality assumption), so it is
robust exactly where the mode/HDI are weakest.

In [ ]:
qs = quantiles(frame, [0.05, 0.5, 0.95])         # (N, 3)
thresholds = [0.0, 1.0, 5.0, 10.0]
exc = exceedance(frame, thresholds)              # (N, 4)

print(f"{'shape':24s}{'q05':>7s}{'q50':>7s}{'q95':>7s} | " +
      "".join(f"P(>{c:g})".rjust(9) for c in thresholds))
for i, name in enumerate(zoo.names):
    print(f"{name:24s}" + "".join(f"{qs[i, j]:7.2f}" for j in range(3)) + " | " +
          "".join(f"{exc[i, j]:9.2f}" for j in range(len(thresholds))))
print("\nonset P(Y>0): high for active shapes, ~", round(float(exc[0, 0]), 2),
      "for the zero-inflated cell; exceedance is non-increasing in the threshold.")

## 6 · Worst-case — `expected_shortfall` (tail mean / CVaR)

`expected_shortfall(frame, tails)` is the mean of the worst `⌈t·S⌉` draws — a *coherent*
worst-case, deliberately **not** the `max` (the high-variance summary it replaces). It is
non-decreasing as the tail deepens (`t → 0`).

**The caveat that matters (small-tail → max).** When `⌈t·S⌉ = 1` the "tail mean"
collapses to the single worst draw — i.e. the `max` you were trying to avoid. That happens
whenever `t` is smaller than `1/S`. At `S=32`, a `t=0.01` tail is *one draw*. Always read
ES with `S` in mind.

In [ ]:
tails = [1.0, 0.5, 0.1, 0.01]
es = expected_shortfall(frame, tails)            # (N, 4), S=1024
print(f"S=1024  {'shape':22s}" + "".join(f"ES(t={t:g})".rjust(12) for t in tails))
for i, name in enumerate(zoo.names):
    print(f"        {name:22s}" + "".join(f"{es[i, j]:12.2f}" for j in range(len(tails))))

# small-tail -> max degeneracy at low S
es_small = expected_shortfall(low.prediction, [0.01])[:, 0]   # S=32 -> ceil(0.01*32)=1
mx = low.prediction.values.max(axis=1)
print("\nS=32, t=0.01  =>  ceil(t*S)=1  =>  ES == max(draws):",
      bool(np.allclose(es_small, mx)))

## 7 · The bimodality flag

`bimodality(frame)` is a deliberately **conservative** 0/1 flag — tuned for *zero false
positives* on the normal (unimodal, skewed, zero-inflated) regime, at the cost of recall
on ambiguous mixtures. Its job is to warn a consumer when a single point / interval would
**hide a second mode**, so you reach for it instead of silently reporting a misleading tip.

In [ ]:
flag = bimodality(frame)[:, 0]
for i, name in enumerate(zoo.names):
    print(f"{name:24s} bimodal={int(flag[i])}")
print("\nonly the genuinely two-peaked shape is flagged; the skewed/zero-inflated "
      "shapes read as unimodal (no crying wolf).")

## 8 · Cross-level aggregation — `HDI(sum) ≠ sum(HDI)`

`aggregate_distributions` sums the per-cell sample arrays **preserving the sample index**
(joint sampling), then you summarize the *aggregate distribution*. You must **aggregate
then summarize**, never summarize then add — the HDI of the country total is not the sum of
the cells' HDIs (joint uncertainty does not add linearly).

In [ ]:
sc = syn.cm_pgm_scenario(n_samples=512, seed=0, lattice=(2, 2), n_months=1)
pgm = sc.pgm_prediction
country = aggregate_distributions(pgm, sc.mapping, SpatialLevel.CM)   # joint-sampled sum

# country 70's constituent grid cells (this month)
cm_units = pgm.index.cross_level_align_arrays(sc.map_keys, sc.map_vals, SpatialLevel.CM).unit
rows70 = np.where(cm_units == 70)[0]
crow = int(np.where(country.index.unit == 70)[0][0])

agg_hdi = hdi(country, mass=0.9)[crow]                 # HDI of the joint sum
sum_of_hdis = hdi(pgm, mass=0.9)[rows70].sum(axis=0)   # naive sum of per-cell HDIs
print("country 70, 90% interval:")
print(f"  HDI(sum)  (correct, joint) : [{agg_hdi[0]:8.1f}, {agg_hdi[1]:8.1f}]  "
      f"width {agg_hdi[1]-agg_hdi[0]:7.1f}")
print(f"  sum(HDI)  (wrong, naive)   : [{sum_of_hdis[0]:8.1f}, {sum_of_hdis[1]:8.1f}]  "
      f"width {sum_of_hdis[1]-sum_of_hdis[0]:7.1f}")
print("  -> summing per-cell intervals OVER-states the joint interval width.")

## 9 · The laws — `assert_summarizer_contract`

The summarize package *publishes* its contract: point estimates return same-type `(N,…,1)`
frames; intervals align to the frame's rows; exceedance is a `[0,1]` survival function;
expected-shortfall lies in `[min, max]`, deepens monotonically and dominates its quantile;
the tower nests. A consumer re-runs this in its own CI.

In [ ]:
assert_summarizer_contract(frame)
print("all summarizer laws hold on the zoo frame ✓")

---
## 🔬 Are the intervals *honest*? (panel additions)

The sections above show *how* to compute each summary. A sharp reviewer (Gelman /
Gneiting / Hyndman / Wilke / the FAO analyst) would next ask *whether they can be
trusted*. Because our data has a **known** DGP, we can check.

### A · Calibration & coverage + PIT, and recovery (register C-59)

**Coverage.** If we draw many independent posteriors *and* a held-out actual from the same
truth, the 90% HDI should contain the actual ~90% of the time. We check the 50/90/95/99
levels. **Calibration is judged *subject to sharpness*** — a uselessly wide interval is
"calibrated" trivially, so coverage is only half the story.

In [ ]:
def coverage(truth, levels, R=600, S=256, seed=0, shrink=1.0):
    """Empirical coverage of each HDI level over R replica posteriors + actuals.

    shrink<1 narrows each posterior toward its median -> a deliberately MIScalibrated
    (over-confident) forecaster, to contrast against the calibrated shrink=1.0 case.
    """
    rng = np.random.default_rng(seed)
    post = np.stack([truth.draw(S, rng) for _ in range(R)]).astype(np.float32)
    if shrink != 1.0:
        med = np.median(post, axis=1, keepdims=True)
        post = (med + shrink * (post - med)).astype(np.float32)
    actuals = np.array([truth.draw(1, rng)[0] for _ in range(R)], np.float32)
    fr = syn.frame_from_samples(post)
    return {lv: float(((actuals >= hdi(fr, mass=lv)[:, 0]) &
                       (actuals <= hdi(fr, mass=lv)[:, 1])).mean()) for lv in levels}

levels = [0.5, 0.9, 0.95, 0.99]
truth = syn.distribution_zoo(seed=0)["right_skewed_gamma"][1]
cal = coverage(truth, levels, shrink=1.0)
mis = coverage(truth, levels, shrink=0.6)
print(f"{'nominal':>10s}{'calibrated':>14s}{'over-confident':>16s}")
for lv in levels:
    print(f"{lv:10.2f}{cal[lv]:14.2f}{mis[lv]:16.2f}")
print("\ncalibrated coverage tracks the nominal level; the over-confident "
      "forecaster under-covers (its intervals are too narrow).")

**PIT histogram.** The probability-integral transform `PIT = F_post(actual)` is
`Uniform(0,1)` iff the forecaster is calibrated. A `∩` shape ⇒ over-dispersed, a `∪` shape
⇒ over-confident. The same calibrated-vs-over-confident pair:

In [ ]:
def pit(truth, R=3000, S=256, seed=1, shrink=1.0):
    rng = np.random.default_rng(seed)
    out = np.empty(R)
    for r in range(R):
        post = truth.draw(S, rng)
        if shrink != 1.0:
            m = np.median(post); post = m + shrink * (post - m)
        out[r] = (post < truth.draw(1, rng)[0]).mean()
    return out

fig, axes = plt.subplots(1, 2, figsize=(8, 2.6), sharey=True)
for ax, (lab, sh) in zip(axes, [("calibrated", 1.0), ("over-confident", 0.6)]):
    ax.hist(pit(truth, shrink=sh), bins=12, range=(0, 1), color="slateblue", density=True)
    ax.axhline(1.0, color="k", ls=":", lw=1)
    ax.set_title(f"PIT — {lab}", fontsize=9); ax.set_xlabel("PIT", fontsize=8)
fig.suptitle("PIT ~ Uniform(0,1) iff calibrated (flat = good; U = over-confident)", fontsize=10)
plt.tight_layout(); plt.show()

**Recovery.** As `S` grows, the point estimate should converge to the known truth.
The tower tip's error vs the true mode shrinks with sample size (posterior-predictive
sanity — a model that can't recover the truth on synthetic data won't on real data):

In [ ]:
truth_g = syn.distribution_zoo(seed=0)["right_skewed_gamma"][1]
Ss = [8, 16, 32, 64, 128, 256, 512, 1024]
errs = []
for S in Ss:
    rng = np.random.default_rng(7)
    post = np.stack([truth_g.draw(S, rng) for _ in range(300)]).astype(np.float32)
    tip = tower_point(syn.frame_from_samples(post)).values[:, 0]
    errs.append(np.abs(tip - truth_g.mode).mean())

fig, ax = plt.subplots(figsize=(5, 2.6))
ax.plot(Ss, errs, "o-"); ax.set_xscale("log", base=2)
ax.set_xlabel("sample count S"); ax.set_ylabel("mean |tip − true mode|")
ax.set_title("recovery of the mode vs S", fontsize=10)
plt.tight_layout(); plt.show()

### B · Equal-tailed vs HDI, and the point-estimate family

On a right-skewed posterior the **equal-tailed** interval (`quantiles` at `5/95`) and the
**HDI** (`hdi` at 0.9) cover the same mass but are *different intervals* — the HDI is
shorter and shifted toward the dense left. Shown with the mean / median / MAP / tower tip
all marked, so you can see how far apart the "central value" candidates sit.

In [ ]:
i = zoo.names.index("right_skewed_gamma")
et = quantiles(frame, [0.05, 0.95])[i]            # equal-tailed 90%
hd = hdi(frame, mass=0.9)[i]                       # HDI 90%
pts = {"mean": est["mean"][i], "median": est["median"][i],
       "MAP": est["map_estimate"][i], "tower tip": est["tower_point"][i]}

fig, ax = plt.subplots(figsize=(8, 2.8))
ax.hist(frame.values[i], bins=50, density=True, color="lightgray")
ax.plot(et, [0.02, 0.02], lw=4, color="darkorange", label=f"equal-tailed (w={et[1]-et[0]:.1f})")
ax.plot(hd, [0.05, 0.05], lw=4, color="steelblue", label=f"HDI 90% (w={hd[1]-hd[0]:.1f})")
for k, v in pts.items():
    ax.axvline(v, lw=1.2, ls="--"); ax.text(v, 0.30, k, rotation=90, fontsize=7, va="top")
ax.legend(fontsize=8); ax.set_yticks([])
ax.set_title("right-skewed gamma: equal-tailed vs HDI; the point-estimate family", fontsize=10)
plt.tight_layout(); plt.show()
print(f"HDI is {(et[1]-et[0]) - (hd[1]-hd[0]):.1f} narrower than the equal-tailed interval "
      "at the same 90% mass.")

### C · Sample-size / Monte-Carlo error

Every summary is itself a random variable in `S`. At `S=32` the point estimate scatters;
by `S=1024` it has settled. The spread of `tower_point` across replicas, by `S`:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2.8))
for S in [8, 32, 128, 1024]:
    rng = np.random.default_rng(3)
    post = np.stack([truth_g.draw(S, rng) for _ in range(400)]).astype(np.float32)
    tips = tower_point(syn.frame_from_samples(post)).values[:, 0]
    ax.hist(tips, bins=30, density=True, alpha=0.5, label=f"S={S}")
ax.axvline(truth_g.mode, color="k", ls="--", lw=1, label="true mode")
ax.legend(fontsize=8); ax.set_yticks([])
ax.set_title("Monte-Carlo error of the tower tip shrinks with S", fontsize=10)
plt.tight_layout(); plt.show()

### D · When each summary misleads

A compact failure gallery — each summary on the shape where it is **most misleading**:
the MAP on a bimodal posterior (lands in one peak, ignores the other); the **mean** on a
heavy tail (dragged right of the bulk, representing no likely outcome); expected-shortfall
at a tiny tail (collapses to the volatile `max`).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 2.8))
# MAP on bimodal
ib = zoo.names.index("bimodal")
axes[0].hist(frame.values[ib], bins=50, density=True, color="lightgray")
axes[0].axvline(est["map_estimate"][ib], color="crimson", lw=2, label="MAP")
axes[0].set_title("MAP on a bimodal posterior\n(one peak, hides the other)", fontsize=8.5)
axes[0].legend(fontsize=8)
# mean on heavy tail
ih = zoo.names.index("heavy_tailed")
axes[1].hist(frame.values[ih], bins=60, density=True, color="lightgray")
axes[1].axvline(est["mean"][ih], color="crimson", lw=2, label="mean")
axes[1].axvline(est["median"][ih], color="seagreen", lw=2, label="median")
axes[1].set_xlim(0, np.quantile(frame.values[ih], 0.99))
axes[1].set_title("mean on a heavy tail\n(right of the bulk)", fontsize=8.5)
axes[1].legend(fontsize=8)
# ES tiny tail -> max
es_curve = expected_shortfall(low.prediction, [0.5, 0.2, 0.1, 1/32])[ih]
axes[2].plot([0.5, 0.2, 0.1, 1/32], es_curve, "o-")
axes[2].axhline(low.prediction.values[ih].max(), color="crimson", ls="--", label="max draw")
axes[2].set_xlabel("tail fraction t (S=32)", fontsize=8)
axes[2].set_title("ES at a tiny tail -> max\n(t<1/S = one draw)", fontsize=8.5)
axes[2].legend(fontsize=8)
for a in axes[:2]: a.set_yticks([])
plt.tight_layout(); plt.show()

### F · A spatial / map view (register C-61)

Per-cell summaries have spatial structure an analyst needs to *see*. On a **toy lattice**
(synthetic coordinates — no real geography, ADR-001), the point estimate and the interval
width as small-multiple maps. Width is itself a map: where is the forecast *uncertain*?

In [ ]:
msc = syn.cm_pgm_scenario(n_samples=256, seed=0, lattice=(6, 6), n_months=1)
mpgm = msc.pgm_prediction
nr, nc = msc.lattice_shape
tip = tower_point(mpgm).values[:, 0]
band = hdi(mpgm, mass=0.9)
width = band[:, 1] - band[:, 0]

def to_grid(vals):
    g = np.full((nr, nc), np.nan)
    for row, pg in enumerate(mpgm.index.unit.tolist()):
        r, c = msc.coords[int(pg)]; g[r, c] = vals[row]
    return g

fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
for ax, (lab, vals) in zip(axes, [("tower tip (point)", tip), ("90% HDI width (uncertainty)", width)]):
    im = ax.imshow(to_grid(vals), cmap="magma"); ax.set_title(lab, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([]); fig.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle("toy lattice — synthetic coordinates, illustrative only", fontsize=10)
plt.tight_layout(); plt.show()

### G · Decision relevance

The operationally-relevant readout for an FAO-style consumer is not the point — it is a
**probability against a threshold**. Onset `P(Y>0)` and a severe-scenario exceedance
`P(Y>c)`, mapped, with a simple alert rule (`P > 0.5`). This is the summary a
food-security decision actually keys on.

In [ ]:
onset = exceedance(mpgm, [0.0])[:, 0]
severe_c = float(np.quantile(mpgm.values, 0.85))      # a synthetic "severe" level
severe = exceedance(mpgm, [severe_c])[:, 0]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, (lab, vals) in zip(
    axes,
    [("onset  P(Y>0)", onset),
     (f"severe  P(Y>{severe_c:.1f})", severe),
     ("alert  P(severe)>0.5", (severe > 0.5).astype(float))],
):
    im = ax.imshow(to_grid(vals), cmap="viridis", vmin=0, vmax=1)
    ax.set_title(lab, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle("decision-relevant exceedance maps (synthetic thresholds, illustrative)", fontsize=10)
plt.tight_layout(); plt.show()
print(f"cells on alert (severe exceedance > 0.5): {int((severe > 0.5).sum())} / {mpgm.n_rows}")

## The summaries, and their honesty — in one screen

| Summary | Reach for it when… | It misleads when… |
|---|---|---|
| `collapse(np.mean)` | you need an additive, conservable quantity | heavy tails drag it off the bulk (§D) |
| `map_estimate` / `tower_point` | you want the "most likely" value | the posterior is multimodal; MAP is low-`S` biased (§3, C-32) |
| `hdi_tower` | you want the *shortest* credible interval | you actually wanted equal-tailed (§B) |
| `quantiles` / `exceedance` | you need `P(Y>c)`, onset, decisions | — (distribution-agnostic; the robust workhorse) |
| `expected_shortfall` | you need a coherent worst-case | `t < 1/S` collapses it to `max` (§6) |
| `bimodality` | to *guard* a point/interval | (conservative; misses subtle bumps by design) |

And the meta-summary: **an interval is only as good as its coverage** (§A). Next:
[`03_reconciliation.ipynb`](03_reconciliation.ipynb) takes these grid posteriors and makes
them sum, per draw, to country totals — and asks whether doing so *helps*.